# 234 - Does a discrete taxonomy survive its own assumptions?

**Figure F2 of the plan.** This notebook fits nothing new - it interrogates the shipped K=5 run. If the answer here is *yes, the partition is stable*, the rest of the plan is unnecessary and should be abandoned.

1. Does the partition survive a change of preprocessing?
2. Is K identifiable?
3. Is the structure spread across clusters, or carried by one?

In [ ]:
import sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
sys.path.insert(0, "functions")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import lf_decompose as D

RUN = Path("outputs/clustering/kmeans/concat_hg/runs/20260803_175417")
OUT = Path("outputs/clustering/decomposition"); OUT.mkdir(parents=True, exist_ok=True)

X    = np.load(RUN / "X_train.npy").astype(np.float64)
lab  = pd.read_csv(RUN / "labels.csv")
CCOL = next(c for c in lab.columns if c.startswith("cluster_") and not c.endswith("_ranked"))
y5   = lab[CCOL].to_numpy()
pat  = lab["patient_id"].astype(str).to_numpy()

# fsaverage coordinates, for anatomical coherence
_co = pd.read_csv("outputs/250_recon/fsaverage/coords/ALL_PATIENTS_contacts_fsaverage.csv")
_nz = lambda s: str(s).replace("_", "").replace("-", "").upper()
_co["key"] = [f"{p}|{_nz(x)}" for p, x in zip(_co["patient"], _co["name"])]
lab["key"] = [f"{p}|{_nz(e)}" for p, e in zip(lab["patient_id"], lab["electrode"])]
XYZ = lab.merge(_co[["key", "x", "y", "z"]], on="key", how="left")[["x", "y", "z"]].to_numpy()

CONDS = ["audio", "picture", "reading"]; NT = X.shape[1] // 3
print(f"{X.shape[0]} electrodes x {X.shape[1]} features | {len(np.unique(pat))} patients "
      f"| {np.isnan(XYZ).any(1).sum()} without coordinates")

## 1 - Does the partition survive a change of preprocessing?

Same data, same K, same seed - only the scaling differs. Adjusted Rand Index: 1 = identical, 0 = no better than chance.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score as ari
from sklearn.decomposition import PCA

variants = {name: D.apply_pipeline(X, name, pat) for name in D.PIPELINES}
variants["pca20"] = PCA(20, random_state=0).fit_transform(X)

fits = {k: KMeans(n_clusters=5, n_init=20, random_state=42).fit_predict(v)
        for k, v in variants.items()}
names = list(fits)
M = pd.DataFrame([[ari(fits[a], fits[b]) for b in names] for a in names],
                 index=names, columns=names)
display(M.round(2))
off = M.where(~np.eye(len(M), dtype=bool))
print("lowest agreement between two defensible pipelines:", round(off.min().min(), 3))

## 2 - Is K identifiable?

Silhouette across K, with the spread from subsampling 80% of electrodes. If the ribbons overlap, K was not chosen by the data.

In [ ]:
from sklearn.metrics import silhouette_score
rng = np.random.default_rng(0); KS = range(3, 9); rows = []
Xs = D.unit_norm(X)
for k in KS:
    for rep in range(12):
        idx = rng.choice(len(Xs), int(0.8 * len(Xs)), replace=False)
        yy = KMeans(n_clusters=k, n_init=10, random_state=rep).fit_predict(Xs[idx])
        rows.append(dict(k=k, rep=rep, sil=silhouette_score(Xs[idx], yy)))
sw = pd.DataFrame(rows)
g = sw.groupby("k")["sil"].agg(["mean", "std"])
display(g.round(4))
best = g["mean"].idxmax()
near = [int(k) for k in g.index if abs(g.loc[k, "mean"] - g.loc[best, "mean"]) < g.loc[best, "std"]]
print(f"best mean K = {best}; indistinguishable from it: {near}")

## 3 - Is the structure spread across clusters, or carried by one?

Per-cluster silhouette mass. A partition where one cluster supplies nearly all the separation is not five types - it is one type and a remainder.

In [ ]:
from sklearn.metrics import silhouette_samples
s = silhouette_samples(X, y5)
tab = pd.DataFrame({"n": pd.Series(y5).value_counts().sort_index(),
                    "mean_sil": pd.Series(s).groupby(y5).mean(),
                    "sil_mass": pd.Series(s).groupby(y5).sum()})
tab["share_of_total_%"] = (100 * tab["sil_mass"] / tab["sil_mass"].sum()).round(1)
display(tab.round(3))
big = tab["sil_mass"].idxmax()
print(f"total silhouette {s.mean():.3f}; without cluster {big}: {s[y5 != big].mean():.3f}")

## 4 - Anatomical coherence per pipeline

Of an electrode's 10 nearest neighbours in fsaverage space, how many share its label - divided by a label shuffle. **This is the criterion to select on: does the partition respect the brain?**

In [ ]:
for name in fits:
    obs, ratio = D.spatial_coherence(fits[name], XYZ)
    print(f"  {name:16s} coherence {obs:.3f}  = {ratio:.2f}x chance")

### Read-out

Low agreement between pipelines + unresolvable K + separation carried by one cluster means a hard partition is a *choice*, not a finding - and the graded decomposition in **235** is the honest alternative.